<h2><b>WP13 - Synthetic data - Privacy risk assessment</h2>

<h3><b> 1. Building of Dataset <i>D</i></h3>

<h4>
  The dataset <i>D</i> has been created artificially with fake data close to the real-world data. To compute the distribution of the following variables: age, civil status, and gender, we refer to
  <a href="https://demo.istat.it/app/?i=POS&a=2023&l=en" target="_blank">https://demo.istat.it/app/?i=POS&a=2023&l=en</a>.
</h4>
<P><h4>In this experimentation we assume it as the real dataset</h4>



# 0. Data Download

In [ ]:
#download data in folder work/data
!../sspcloud/download_data.sh

/bin/bash: line 1: ../sspcloud/download_data.sh: No such file or directory


In [2]:
import os

step1 = "../../data/Step1/"
if not os.path.exists(step1 + "Input"):
    os.makedirs(step1 + "Input")
if not os.path.exists(step1 + "Output"):
    os.makedirs(step1 + "Output")

!mv ../../data/*.* {step1}Input

# 1. Importing required packages

In [3]:
import pandas as pd
import numpy as np
import os
import random
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import zipfile
from sklearn.preprocessing import MinMaxScaler
from matplotlib.ticker import FuncFormatter
from datetime import datetime, timedelta
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

In [4]:
#username = 'donatella.papa'
base_dir = rf"../../data/Step1"
input_dir = os.path.join(base_dir, "Input")
os.chdir(input_dir)

# 2. Set General Variables and Paths

In [5]:
# Reference date of the selected sample: 1st January 2023
reference_date = datetime(2023, 1, 1)

# Number of records to generate
number_records = 10000

# Weight to assign to 'age' in the categorization formula (1 to 100, recommended below 35)
thresholdValue = 20

# Flag to apply random perturbation (Y/N)
R = "N"

# Percentage of random flipping, if R == "Y"
if R != "N":
    PR = 40
else:
    PR = 0

# List of Italian municipality codes (source: Istat). Used as proxy for birth place.
list_municipality_names = [
    "Agrigento", "Alessandria", "Ancona", "Aosta", "L'Aquila", "Arezzo", "Ascoli-Piceno",
    "Asti", "Avellino", "Bari", "Barletta-Andria-Trani", "Belluno", "Benevento", "Bergamo",
    "Biella", "Bologna", "Bolzano", "Brescia", "Brindisi", "Cagliari", "Caltanissetta",
    "Campobasso", "Carbonia-Iglesias", "Caserta", "Catania", "Catanzaro", "Chieti", "Como",
    "Cosenza", "Cremona", "Crotone", "Cuneo", "Enna", "Fermo", "Ferrara", "Firenze",
    "Foggia", "Forli-Cesena", "Frosinone", "Genova", "Gorizia", "Grosseto", "Imperia",
    "Isernia", "La-Spezia", "Latina", "Lecce", "Lecco", "Livorno", "Lodi", "Lucca",
    "Macerata", "Mantova", "Massa-Carrara", "Matera", "Medio-Campidano", "Messina",
    "Milano", "Modena", "Monza-Brianza", "Napoli", "Novara", "Nuoro", "Ogliastra",
    "Olbia-Tempio", "Oristano", "Padova", "Palermo", "Parma", "Pavia", "Perugia",
    "Pesaro-Urbino", "Pescara", "Piacenza", "Pisa", "Pistoia", "Pordenone", "Potenza",
    "Prato", "Ragusa", "Ravenna", "Reggio-Calabria", "Reggio-Emilia", "Rieti", "Rimini",
    "Roma", "Rovigo", "Salerno", "Sassari", "Savona", "Siena", "Siracusa", "Sondrio",
    "Taranto", "Teramo", "Terni", "Torino", "Trapani", "Trento", "Treviso", "Trieste",
    "Udine", "Varese", "Venezia", "Verbania", "Vercelli", "Verona", "Vibo-Valentia",
    "Vicenza", "Viterbo"
]

# Set the seed for reproducibility
np.random.seed(42)
random_seed = random.seed(42)

# 3.  Load and Process Distributions (Age, Civil Status, Gender, Municipality)

In [6]:
# Load resident population data
dfload = pd.read_csv('Resident population.csv', delimiter=',')

# Convert 'Age' column to numeric
dfload['Age'] = pd.to_numeric(dfload['Age'], errors='coerce')
dfload = dfload.dropna(subset=['Age'])
dfload['Age'] = dfload['Age'].astype(int)

# Aggregate civil status columns
dfload['NeverMarried'] = dfload['Never married males'] + dfload['Never married females']
dfload['Married'] = dfload['Married males'] + dfload['Married females']
dfload['Divorced'] = dfload['Divorced males'] + dfload['Divorced females']
dfload['Widowed'] = dfload['Widowed males'] + dfload['Widowed females']
dfload['CivilPartner'] = dfload['Same sex civil partner males'] + dfload['Same sex civil partner females']
dfload['Total'] = dfload['Total males'] + dfload['Total females']

# Filter records for age between 18 and 65
df = dfload.loc[(dfload['Age'] >= 18) & (dfload['Age'] <= 65)]
df = df[['Total males', 'Total females', 'Total', 'NeverMarried', 'Married', 'Divorced', 'Widowed', 'CivilPartner', 'Age']]

# Calculate distributions
total_civil_status = df[['NeverMarried', 'Married', 'Divorced', 'Widowed', 'CivilPartner']].sum(axis=0)
total_civil_status_distribution = total_civil_status / total_civil_status.sum()

total_gender = df[['Total males', 'Total females']].sum(axis=0)
total_gender_distribution = total_gender / total_gender.sum()

age_distribution = df.groupby('Age')['Total'].sum().reset_index()
age_distribution['distribution'] = age_distribution['Total'] / age_distribution['Total'].sum()
age_distribution = age_distribution.set_index('Age')['distribution'].to_dict()

# Load municipality data from ZIP file
zip_file_path = 'POSAS_2023_en_Municipalities.zip'
csv_file_name = 'POSAS_2023_en_Municipalities.csv'
dtype_dict = {'Municipality code': str}

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    with zip_ref.open(csv_file_name) as csv_file:
        df_mun = pd.read_csv(csv_file, skiprows=1, dtype=dtype_dict)

# Filter for municipalities in Brindisi, Lecce, Taranto and for age 18-65
filtered_df = df_mun.loc[(df_mun['Age'] >= 18) & (df_mun['Age'] <= 65)]
filtered_df = filtered_df[filtered_df['Municipality code'].str.startswith(('073', '074', '075'))]
filtered_df['Total'] = filtered_df['Total males'] + filtered_df['Total females']
selected_columns = ['Municipality code', 'Municipality', 'Age', 'Total']
final_df = filtered_df.loc[:, selected_columns]

# Calculate municipality distribution
df_municipality_distribution = final_df.groupby(['Municipality code', 'Municipality'])['Total'].sum().reset_index()
df_municipality_distribution['distribution'] = df_municipality_distribution['Total'] / df_municipality_distribution['Total'].sum()
municipality_distribution = df_municipality_distribution.set_index('Municipality')['distribution'].to_dict()

# 4. Helper Functions for Data Generation

In [7]:
# Generate a random birth date based on age
def generate_birth_date(age):
    birth_year = reference_date.year - age - 1
    birth_month = random.randint(1, 12)

    if birth_month == 2:
        if (birth_year % 4 == 0 and birth_year % 100 != 0) or (birth_year % 400 == 0):
            birth_day = random.randint(1, 29)
        else:
            birth_day = random.randint(1, 28)
    elif birth_month in [4, 6, 9, 11]:
        birth_day = random.randint(1, 30)
    else:
        birth_day = random.randint(1, 31)

    return datetime(birth_year, birth_month, birth_day)

# Generate civil status based on distribution
def generate_civil_status():
    statuses = ['NeverMarried', 'Married', 'Divorced', 'Widowed', 'CivilPartner']
    weights = total_civil_status_distribution.values
    return random.choices(statuses, weights=weights)[0]

# Generate gender based on distribution
def generate_gender():
    genders = ['Male', 'Female']
    weights = total_gender_distribution.values
    return random.choices(genders, weights=weights)[0]

# Calculate day of the year from month and day
def day_of_year(birth_month: int, birth_day: int) -> int:
    days_in_month = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

    if not (1 <= birth_month <= 12):
        raise ValueError("Month must be between 1 and 12.")
    if not (1 <= birth_day <= days_in_month[birth_month - 1]):
        raise ValueError("Day is not valid for the specified month.")

    return sum(days_in_month[:birth_month - 1]) + birth_day

# 5. Core Function to Generate Diagnosis (V4)

In [8]:
def generate_diagnosis_v4(df):
    """Generate a four-class diagnosis based on normalized age, activity, and predisposition."""
    th = (thresholdValue / 100)
    th_complement = (1 - th)

    scaler = MinMaxScaler()

    # Normalize columns
    df['age_norm'] = scaler.fit_transform(df['age'].values.reshape(-1, 1))
    df['activity_norm'] = scaler.fit_transform(df['physical_activity'].values.reshape(-1, 1))
    df['predisposition_norm'] = scaler.fit_transform(df['genetic_predisposition'].values.reshape(-1, 1))

    # Calculate combined scores
    df['score1'] = th * df['age_norm'] + th_complement * (1 - df['activity_norm'])
    df['score2'] = th * df['age_norm'] + th_complement * df['predisposition_norm']

    # Find global medians
    thr1 = df['score1'].median()
    thr2 = df['score2'].median()

    # Binarize
    df['bin1'] = (df['score1'] >= thr1).astype(int)
    df['bin2'] = (df['score2'] >= thr2).astype(int)

    # Combine into 4 classes (1-4)
    df['diagnosis'] = 2 * df['bin1'] + df['bin2'] + 1

    # Add raw class representation
    mapping = {1: "00", 2: "01", 3: "10", 4: "11"}
    df['raw_class'] = df['diagnosis'].map(mapping)

    return df

# 6. Generate the Complete Dataset

In [9]:
# Initialize data dictionary
data = {
    'id': range(1, number_records + 1),
    'municipality_residence': [random.choices(list(municipality_distribution.keys()), weights=list(municipality_distribution.values()))[0] for _ in range(number_records)],
    'age': [random.choices(list(age_distribution.keys()), weights=list(age_distribution.values()))[0] for _ in range(number_records)],
    'civil_status': [generate_civil_status() for _ in range(number_records)],
    'gender': [generate_gender() for _ in range(number_records)],
    'occupation': [random.choices([0, 1], weights=[0.35, 0.65])[0] for _ in range(number_records)],
    'physical_activity': np.random.triangular(1, 2, 6, size=number_records).astype(int),
    'genetic_predisposition': np.random.triangular(1, 4, 9, size=number_records).astype(int),
}

# Create DataFrame
dataset = pd.DataFrame(data)

# Generate birth dates from ages
dataset['birth_date'] = dataset['age'].apply(generate_birth_date)

# Split birth_date into year, month, day
dataset['birth_year'] = dataset['birth_date'].dt.year
dataset['birth_month'] = dataset['birth_date'].dt.month
dataset['birth_day'] = dataset['birth_date'].dt.day
dataset.drop(columns=['birth_date'], inplace=True)

# Generate the diagnosis column
dataset = generate_diagnosis_v4(dataset)

# Generate municipality of birth (50% same as residence, 50% random from list)
dataset['municipality_birth'] = [
    municipality if random.random() < 0.5 else random.choice(list_municipality_names)
    for municipality in dataset['municipality_residence']
]

# Calculate day of year for birth
dataset["birth_dayofyear"] = dataset.apply(
    lambda row: day_of_year(row["birth_month"], row["birth_day"]),
    axis=1
)


# 7. Save the Dataset

In [10]:
# Create output directory dynamically
output_dir = os.path.join(base_dir, "Output")
#os.makedirs(output_dir, exist_ok=True)

# Save to CSV
number_records_file = round(number_records / 1000)
output_file = f'real_data_datasetM{number_records_file}_TH{thresholdValue}_R{R}_PR{PR}_4CAT_.csv'
print(f"Dataset generated successfully in {output_dir}.")
os.chdir("../")

Dataset generated successfully in ../../data/Step1/Output.


# 8. Missing on X2 features (physical_activity and genetic predisposition)

In [11]:
# Imposta il seme per la riproducibilità
np.random.seed(42)

# Assicurati che le colonne siano stringhe
dataset['physical_activity'] = dataset['physical_activity'].astype(str)
dataset['genetic_predisposition'] = dataset['genetic_predisposition'].astype(str)

# Scegli un indice casuale per ciascuna colonna
missing_index_pa = np.random.choice(dataset.index, size=1, replace=False)
missing_index_gp = np.random.choice(dataset.index, size=1, replace=False)

# Imposta "Missing" in una sola cella per colonna
dataset.loc[missing_index_pa, 'physical_activity'] = 'Missing'
dataset.loc[missing_index_gp, 'genetic_predisposition'] = 'Missing'

# Save the dataset in a CSV file
os.chdir(output_dir) 
dataset.to_csv(f'real_data_datasetM{number_records_file}_TH{thresholdValue}_R{R}_PR{PR}_4CAT_MISS_X2.csv', index=False)


print("Dataset generate successfully.")
display(dataset)

# Conta la distribuzione dei valori nella colonna 'diagnosis'
distribution = dataset['diagnosis'].value_counts()

# Mostra la distribuzione
print(distribution)

grouped_df = dataset.groupby(['age', 'diagnosis']).size().reset_index(name='count')
print(grouped_df)

# verifica
missing_summary = pd.DataFrame({
    'n_Missing': (dataset == 'Missing').sum(),
    'pct_Missing': (dataset == 'Missing').sum() / len(dataset) * 100
})
print(missing_summary)

Dataset generate successfully.


,id,municipality_residence,age,civil_status,gender,occupation,physical_activity,genetic_predisposition,birth_year,birth_month,...,activity_norm,predisposition_norm,score1,score2,bin1,bin2,diagnosis,raw_class,municipality_birth,birth_dayofyear
0,1,Corigliano d'Otranto,63,Married,Male,0,2,3,1959,11,...,0.25,0.285714,0.791489,0.420061,1,0,3,10,Bari,320
1,2,Faggiano,61,Married,Female,1,5,3,1961,3,...,1.00,0.285714,0.182979,0.411550,0,0,1,00,Cosenza,61
2,3,Taranto,53,NeverMarried,Male,0,3,3,1969,11,...,0.50,0.285714,0.548936,0.377508,0,0,1,00,Taranto,332
3,4,Taranto,56,NeverMarried,Female,1,3,5,1966,4,...,0.50,0.571429,0.561702,0.618845,0,1,2,01,Olbia-Tempio,96
4,5,Lecce,20,Married,Female,0,1,4,2002,12,...,0.00,0.428571,0.808511,0.351368,1,0,3,10,Savona,344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,Surbo,52,NeverMarried,Male,0,4,6,1970,2,...,0.75,0.714286,0.344681,0.716109,0,1,2,01,Surbo,47
9996,9997,Galatina,37,Married,Male,0,4,2,1985,11,...,0.75,0.142857,0.280851,0.195137,0,0,1,00,Galatina,334
9997,9998,Salve,43,NeverMarried,Male,1,4,3,1979,6,...,0.75,0.285714,0.306383,0.334954,0,0,1,00,Salve,169
9998,9999,Taranto,55,Married,Female,1,2,4,1967,4,...,0.25,0.428571,0.757447,0.500304,1,1,4,11,Trapani,101


diagnosis
4    2641
1    2552
2    2406
3    2401
Name: count, dtype: int64
     age  diagnosis  count
0     18          1     84
1     18          2     41
2     18          3     20
3     18          4     10
4     19          1     90
..   ...        ...    ...
187   64          4     80
188   65          1     37
189   65          2     58
190   65          3     42
191   65          4     65

[192 rows x 3 columns]
                        n_Missing  pct_Missing
id                              0         0.00
municipality_residence          0         0.00
age                             0         0.00
civil_status                    0         0.00
gender                          0         0.00
occupation                      0         0.00
physical_activity               1         0.01
genetic_predisposition          1         0.01
birth_year                      0         0.00
birth_month                     0         0.00
birth_day                       0         0.00
age_norm    